In [1]:
import mlflow
mlflow.set_tracking_uri('http://3.111.55.134:5000')

c:\Users\areeb\OneDrive\Desktop\yt_cmmnt_analyzer\yt_comment_analyzer\analyzer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# Set or create an experiment
mlflow.set_experiment("Exp 5 - ML Algos with HP Tuning.")

<Experiment: artifact_location='s3://mlflow-buckets-areeba/10', creation_time=1784308564725, effective_trace_archival_retention=None, experiment_id='10', last_update_time=1784308564725, lifecycle_stage='active', name='Exp 5 - ML Algos with HP Tuning.', tags={}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna

In [6]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for KNN

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigramss")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model - trust scipy sparse matrix type since KNN stores
        # the training data (a sparse TF-IDF matrix) internally
        mlflow.sklearn.log_model(
            model,
            f"{model_name}_model",
            skops_trusted_types=["scipy.sparse._csr.csr_matrix"]
        )


# Step 6: Optuna objective function for KNN
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 3, 30)  # Tuning the number of neighbors
    p = trial.suggest_categorical('p', [1, 2])  # 1 = Manhattan, 2 = Euclidean

    model = KNeighborsClassifier(n_neighbors=n_neighbors, p=p)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for KNN, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_knn, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = KNeighborsClassifier(n_neighbors=best_params['n_neighbors'], p=best_params['p'])

    # Log the best model with MLflow, passing the algo_name as "KNN"
    log_mlflow("KNN", best_model, X_train, X_test, y_train, y_test)


# Run the experiment for KNN
run_optuna_experiment()

[I 2026-07-18 00:03:34,976] A new study created in memory with name: no-name-ec6e89b7-a5f1-4f07-abd7-fe256f60c49c
[I 2026-07-18 00:03:42,686] Trial 0 finished with value: 0.3818431621221729 and parameters: {'n_neighbors': 18, 'p': 1}. Best is trial 0 with value: 0.3818431621221729.
[I 2026-07-18 00:03:49,614] Trial 1 finished with value: 0.5299091101247093 and parameters: {'n_neighbors': 27, 'p': 2}. Best is trial 1 with value: 0.5299091101247093.
[I 2026-07-18 00:03:56,875] Trial 2 finished with value: 0.5619319382794336 and parameters: {'n_neighbors': 12, 'p': 2}. Best is trial 2 with value: 0.5619319382794336.
[I 2026-07-18 00:04:04,454] Trial 3 finished with value: 0.37856689917564995 and parameters: {'n_neighbors': 22, 'p': 1}. Best is trial 2 with value: 0.5619319382794336.
[I 2026-07-18 00:04:10,673] Trial 4 finished with value: 0.5388924117522722 and parameters: {'n_neighbors': 22, 'p': 2}. Best is trial 2 with value: 0.5619319382794336.
[I 2026-07-18 00:04:18,323] Trial 5 fini

🏃 View run KNN_SMOTE_TFIDF_Trigramss at: http://3.111.55.134:5000/#/experiments/10/runs/899a8c60352343708b18790cfed51a5c
🧪 View experiment at: http://3.111.55.134:5000/#/experiments/10
